# energy-audit · Step 3 · Magenta RT 2 (mrt2_base): full generation, 300 clips (version 2: resume)

**Version 2 (2026-11-08).** Version 1 stopped at 66/300: the old non-silence audit rejected a
very quiet clip. Under Conner's decision A, the harness now halts only on format errors, non-finite
samples or digital silence, and flags quiet clips (`quiet_flag`, RMS < -60 dBFS) instead. This
version restores v1's 66 audited clips from its output (attached as input) and generates the other 234.

The frozen matrix (`data/prompts.csv`, 50 prompts) × 6 seeds, 30 s each, generated by `gen/mrt2.py` at
commit `91d5677`, with the environment from `rerun/step3/requirements-gen-mrt2.txt`. The same setup
passed smoke v1/v2: every clip audited, seeds distinct, same seed bit-identical within and across sessions.

**Time stop:** the harness stops cleanly after 11.0 h; everything finished so far is packed into the
Output. **Resume:** push a new version with this notebook's own output attached as an input
(`kernel_sources`). Cell 3 restores the finished clips and the manifest, and the harness skips them.

Output: `audio_mrt2.tar` (WAVs), `manifest_mrt2.csv` (one row per clip), `run_manifest.json`
(model, revision, GPU, image, wall-clock, completeness), `generate_log.txt`, `env.json`, `pip_freeze.txt`.

In [ ]:
# 1. Fetch the harness, frozen matrix and pinned requirements at the commit; verify hashes
import subprocess, sys, os, json, time, socket, platform, hashlib, urllib.request, datetime
T0 = time.time(); STARTED = datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z"
W = "/kaggle/working"; os.makedirs(W, exist_ok=True)
socket.getaddrinfo("huggingface.co", 443); print("internet: OK")
COMMIT = "91d5677fd99a57fd70a4c35bb9fd4f9568d693c0"
WANT = {"gen/mrt2.py": "3bfaec69fcc2e7cf24384dc3dbe9cd5de04da42462bdab767e6b5b22cc0755df",
        "data/prompts.csv": "9bf219753c62e83ecc47181002d5a49d6e83ff15ee7e60a64f572be12f0e2a2e",
        "rerun/step3/requirements-gen-mrt2.txt": "064ebcd3c3e91bbbe9620de202e3e18caeb11cc6a04268c6522233b974501575"}
R = "/tmp/repo"
for rel, h in WANT.items():
    os.makedirs(os.path.dirname(f"{R}/{rel}"), exist_ok=True)
    b = urllib.request.urlopen(f"https://raw.githubusercontent.com/ConnerFarrell/energy-audit/{COMMIT}/{rel}").read()
    assert hashlib.sha256(b).hexdigest() == h, rel
    open(f"{R}/{rel}", "wb").write(b)
import shutil; shutil.copy(f"{R}/data/prompts.csv", f"{W}/prompts.csv")
print("inputs: hash OK at", COMMIT[:7])


In [ ]:
# 2. Pinned install + JAX GPU check (subprocesses; this kernel never imports the new numpy or jax)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{R}/rerun/step3/requirements-gen-mrt2.txt"], check=True)
open(f"{W}/pip_freeze.txt", "w").write(subprocess.run([sys.executable, "-m", "pip", "freeze"],
                                         capture_output=True, text=True).stdout)
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
probe = subprocess.run([sys.executable, "-c",
    "import jax,jaxlib,flax,numpy,magenta_rt,json;d=jax.devices();"
    "x=jax.numpy.ones((256,256));y=float((x@x).sum());"
    "print(json.dumps(dict(jax=jax.__version__,jaxlib=jaxlib.__version__,flax=flax.__version__,numpy=numpy.__version__,"
    "magenta_rt=magenta_rt.__version__,devices=[str(i) for i in d],platform=d[0].platform,matmul_ok=y==256**3)))"],
    capture_output=True, text=True)
print(probe.stdout[-2000:], probe.stderr[-3000:])
ENV = dict(python=platform.python_version(), gpu=smi, requirements="rerun/step3/requirements-gen-mrt2.txt@" + COMMIT[:7],
           kaggle_docker_image=os.environ.get("KAGGLE_DOCKER_IMAGE", "not exposed (pinned in kernel metadata)"))
ENV.update(json.loads(probe.stdout.strip().splitlines()[-1]))
json.dump(ENV, open(f"{W}/env.json", "w"), indent=1); print(json.dumps(ENV, indent=1))
assert ENV["platform"] == "gpu" and ENV["matmul_ok"], "JAX CUDA 12 build does not see the GPU"


In [ ]:
# 3. Resume: restore finished clips + manifest from an earlier version's output, if attached as input
import glob, tarfile, csv
OUTD = "/tmp/gen"; os.makedirs(f"{OUTD}/audio", exist_ok=True)
restored = 0
for t in glob.glob("/kaggle/input/**/audio_mrt2.tar", recursive=True):
    with tarfile.open(t) as tf:
        for m in tf.getmembers():
            b = os.path.basename(m.name)
            if m.isfile() and b.startswith("mrt2__") and b.endswith(".wav") and not os.path.exists(f"{OUTD}/audio/{b}"):
                m.name = b; tf.extract(m, path=f"{OUTD}/audio"); restored += 1
prev = sorted(glob.glob("/kaggle/input/**/manifest_mrt2.csv", recursive=True))
if prev:
    rows = [r for r in csv.DictReader(open(prev[-1])) if os.path.exists(f"{OUTD}/audio/{r['clip']}")]
    with open(f"{OUTD}/manifest_mrt2.csv", "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()) if rows else []); w.writeheader(); w.writerows(rows)
    # A clip on disk without a manifest row is regenerated, not trusted.
    have = {r["clip"] for r in rows}
    for p in glob.glob(f"{OUTD}/audio/*.wav"):
        if os.path.basename(p) not in have: os.remove(p); restored -= 1
print(f"restored from earlier version: {restored} clips")
RESTORED = restored


In [ ]:
# 4. Generate everything not yet done; stop cleanly at 11.0 h (harness clock)
DEADLINE_H = 11.0
with open(f"{W}/generate_log.txt", "a") as log:
    p = subprocess.Popen([sys.executable, f"{R}/gen/mrt2.py", "--prompts", f"{R}/data/prompts.csv",
                          "--out", OUTD, "--magenta-home", "/tmp/magenta", "--size", "mrt2_base",
                          "--deadline-h", str(DEADLINE_H)],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        log.write(line); log.flush()
        if line.startswith("[mrt2]") and ("/300" not in line or line.split()[1].split("/")[0].endswith("0")):
            print(line, end="")
        elif "Error" in line or "Traceback" in line: print(line, end="")
    RC = p.wait()
print("harness exit code:", RC)


In [ ]:
# 5. Verify, pack, write the run manifest (always runs, so a partial run is still saved)
import statistics
M = list(csv.DictReader(open(f"{OUTD}/manifest_mrt2.csv"))) if os.path.exists(f"{OUTD}/manifest_mrt2.csv") else []
wavs = sorted(os.path.basename(p) for p in glob.glob(f"{OUTD}/audio/*.wav"))
P = list(csv.DictReader(open(f"{R}/data/prompts.csv")))
expected = sorted(f"mrt2__{r['prompt_id']}__s{k}.wav" for r in P for k in range(6))
names = [r["clip"] for r in M]
checks = dict(
    clips_on_disk=len(wavs), manifest_rows=len(M), expected=len(expected),
    complete=wavs == expected and sorted(names) == expected,
    manifest_matches_disk=sorted(names) == wavs, manifest_unique=len(set(names)) == len(names),
    all_48k_stereo_30s=all(r["sample_rate"] == "48000" and r["channels"] == "2" and r["n_samples"] == "1440000" for r in M),
    distinct_pcm=len({r["pcm_sha256"] for r in M}), all_pcm_distinct=len({r["pcm_sha256"] for r in M}) == len(M),
    distinct_seeds=len({r["seed"] for r in M}),
    quiet_flagged=[r["clip"] for r in M if r.get("quiet_flag") == "1"],
    partial_leftovers=sorted(os.listdir(f"{OUTD}/partial")) if os.path.isdir(f"{OUTD}/partial") else [],
    min_peak=min((float(r["peak"]) for r in M), default=None),
    min_rms_dbfs=min((float(r["rms_dbfs"]) for r in M), default=None))
with tarfile.open(f"{W}/audio_mrt2.tar", "w") as tf:
    for b in wavs: tf.add(f"{OUTD}/audio/{b}", arcname=f"audio/{b}")
with tarfile.open(f"{W}/audio_mrt2.tar") as tf:
    checks["tar_wavs"] = sum(1 for m in tf.getmembers() if m.name.endswith(".wav"))
shutil.copy(f"{OUTD}/manifest_mrt2.csv", f"{W}/manifest_mrt2.csv") if M else None
g = [float(r["gen_s"]) for r in M]
run = dict(
    model="google/magenta-realtime-2", size="mrt2_base",
    hf_revision="010aa0dcb0dfd27b24f0ad07b4dad63e8f9521cc", library="magenta-rt 2.0.3 (JAX backend)",
    harness=f"gen/mrt2.py @ {COMMIT}", requirements=ENV["requirements"],
    gpu=ENV["gpu"], jax_devices=ENV["devices"], kaggle_docker_image_reported=ENV["kaggle_docker_image"],
    started_utc=STARTED, finished_utc=datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    notebook_wall_h=round((time.time() - T0) / 3600, 3), harness_exit_code=RC,
    restored_from_earlier_version=RESTORED, made_this_session=len(M) - RESTORED,
    gen_s_mean=round(statistics.mean(g), 2) if g else None, gen_s_total_h=round(sum(g) / 3600, 3) if g else None,
    deadline_h=DEADLINE_H, checks=checks,
    status="COMPLETE" if checks["complete"] and checks["all_pcm_distinct"] and RC == 0
           else "PARTIAL: push a new version with this output attached as input to resume")
json.dump(run, open(f"{W}/run_manifest.json", "w"), indent=1); print(json.dumps(run, indent=1))
print(sorted(os.listdir(W)))
